# Codex Ultra Deepnote Launcher

Deepnote-only notebook. Use this file on Deepnote; use `launch.ipynb` on Google Colab.

Why a separate file? Deepnote's base image has no Node.js, so the `cxu` entrypoint
installed by `install.sh` fails with `/usr/bin/env: 'node': No such file or directory`
(exit 127). This notebook installs Node.js 20 LTS first, then installs `cxu`,
then starts the service and exposes it through a public tunnel.

- Default service port: `43110`
- Default tunnel: Cloudflare Quick Tunnel (no account or token required)
- Optional alternatives: `localtunnel` or `ngrok`
- ngrok can use an existing login/configuration; set `NGROK_AUTHTOKEN` only when authentication is needed
- Keep the Deepnote session/project running while you use the public URL

> The public URL is reachable by anyone who has it. Do not expose private projects or credentials.

In [ ]:
import atexit
import os
import platform
import re
import shutil
import signal
import socket
import subprocess
import sys
import time
from pathlib import Path

PORT = int(os.environ.get('PORT', '43110'))
TUNNEL_PROVIDER = os.environ.get('TUNNEL_PROVIDER', 'cloudflare').lower()
REPO_DIR = Path.cwd()
HOME = Path.home()
BIN_DIR = HOME / '.local' / 'bin'
BIN_DIR.mkdir(parents=True, exist_ok=True)
NODE_DIR = HOME / '.local' / 'node'

os.environ.update({
    'CODEX_UI_TRUST_PROXY': '1',
    'CODEX_UI_ALLOWED_ORIGINS': '*',
    'CODEX_UI_HOST': '0.0.0.0',
    'PORT': str(PORT),
    'CODEX_UI_LEGAL_DIR': os.environ.get('CODEX_UI_LEGAL_DIR', str(REPO_DIR)),
    'PATH': f"{HOME}/.codex-ultra/bin:{HOME}/.codex-ultra/runtime/bun/bin:{HOME}/.bun/bin:{BIN_DIR}:{NODE_DIR}/bin:{os.environ.get('PATH', '')}",
})

print(f'Working directory: {REPO_DIR}')
print(f'Codex Ultra port: {PORT}')
print(f'Tunnel provider: {TUNNEL_PROVIDER}')
print('Proxy trust enabled; allowed origins: *')

In [ ]:
def run(command, check=True):
    print('$', ' '.join(map(str, command)))
    return subprocess.run(command, check=check, text=True)


def node_ok(min_major=18):
    node = shutil.which('node')
    if node is None:
        return False
    try:
        out = subprocess.run([node, '--version'], capture_output=True, text=True, timeout=15)
        ver = (out.stdout or out.stderr).strip().lstrip('v')
        major = int(ver.split('.')[0])
        print(f'node found: {node} (v{ver})')
        return major >= min_major
    except Exception as exc:
        print(f'node check failed: {exc}')
        return False


def install_node_apt():
    # Best effort: Deepnote containers are Debian/Ubuntu based and usually allow apt.
    try:
        if shutil.which('apt-get') is None:
            return False
        prefix = ['sudo', '-n'] if shutil.which('sudo') is not None else []
        probe = subprocess.run(prefix + ['apt-get', '--version'], capture_output=True)
        if probe.returncode != 0 and prefix:
            prefix = []
        run(prefix + ['apt-get', 'update'], check=False)
        result = subprocess.run(prefix + ['apt-get', 'install', '-y', 'nodejs', 'npm'], capture_output=True, text=True)
        print((result.stdout or '')[-2000:])
        print((result.stderr or '')[-2000:])
        return result.returncode == 0 and node_ok()
    except Exception as exc:
        print(f'apt install failed: {exc}')
        return False


def install_node_tarball(version='20.19.0'):
    machine = platform.machine().lower()
    node_arch = {'x86_64': 'x64', 'amd64': 'x64', 'aarch64': 'arm64', 'arm64': 'arm64'}.get(machine)
    if node_arch is None:
        raise RuntimeError(f'Unsupported CPU architecture for Node.js: {machine}')
    name = f'node-v{version}-linux-{node_arch}'
    url = f'https://nodejs.org/dist/v{version}/{name}.tar.xz'
    archive = Path('/tmp') / f'{name}.tar.xz'
    run(['curl', '-fL', '--retry', '3', '-o', str(archive), url])
    NODE_DIR.mkdir(parents=True, exist_ok=True)
    run(['tar', '-xJf', str(archive), '-C', '/tmp'])
    extracted = Path('/tmp') / name
    for child in extracted.iterdir():
        target = NODE_DIR / child.name
        if target.exists() or target.is_symlink():
            import shutil as _shutil
            if target.is_dir() and not target.is_symlink():
                _shutil.rmtree(target)
            else:
                target.unlink()
        child.rename(target)
    for exe in ('node', 'npm', 'npx'):
        link = BIN_DIR / exe
        real = NODE_DIR / 'bin' / exe
        if real.exists():
            if link.exists() or link.is_symlink():
                link.unlink()
            link.symlink_to(real)
    return node_ok()


if not node_ok():
    print('Node.js 18+ not found; installing Node.js 20 LTS for Deepnote...')
    if not install_node_apt():
        print('apt path did not provide Node.js; falling back to nodejs.org tarball...')
        install_node_tarball()

if not node_ok():
    raise RuntimeError("Node.js 18+ is required but could not be installed. See output above.")

print('node:', shutil.which('node'))
print('npm:', shutil.which('npm') or 'not found')
print('npx:', shutil.which('npx') or 'not found')
print('bun:', shutil.which('bun') or 'not found yet (installed with cxu below)')

In [ ]:
# Node.js is ready at this point, so install.sh can provision Bun + cxu reliably.
if shutil.which('cxu') is None:
    run(['bash', '-lc', 'curl -fsSL https://install.codex-ultra.top/install.sh | sh'])

if shutil.which('codex') is None:
    print('Codex CLI not found; installing the standalone runtime...')
    run(['bash', '-lc', 'curl -fsSL https://chatgpt.com/codex/install.sh | sh'], check=False)

print('cxu:', shutil.which('cxu') or 'not found')
print('codex:', shutil.which('codex') or 'not found')
print('bun:', shutil.which('bun') or 'not found')
print('node:', shutil.which('node') or 'MISSING')
if shutil.which('node') is None:
    raise RuntimeError("node is still missing after install; 'cxu serve' would fail with exit 127")

In [ ]:
service_process = None
tunnel_process = None
public_url = None
log_path = '/tmp/codex-ultra-service.log'

def port_is_open(host='127.0.0.1', port=PORT):
    with socket.socket() as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0


def dump_log_tail(lines=60):
    try:
        with open(log_path, encoding='utf-8', errors='replace') as handle:
            tail = handle.readlines()[-lines:]
        print(f'--- last {len(tail)} lines of {log_path} ---')
        for line in tail:
            print(line, end='')
        print(f'--- end of {log_path} ---')
    except Exception as exc:
        print(f'Could not read service log: {exc}')


def candidate_commands():
    candidates = []
    cxu = shutil.which('cxu')
    if cxu:
        candidates.append([cxu, 'serve', '--port', str(PORT)])
    bun = shutil.which('bun')
    if bun:
        candidates.append([bun, 'x', '@codex-ultra/cxu', 'serve', '--port', str(PORT)])
    home_bun = HOME / '.codex-ultra' / 'runtime' / 'bun' / 'bin' / 'bun'
    if home_bun.is_file():
        candidates.append([str(home_bun), 'x', '@codex-ultra/cxu', 'serve', '--port', str(PORT)])
    bundle_bun = Path('/opt/codex-ultra/runtime/bun/bin/bun')
    bundle_js = Path('/opt/codex-ultra/server-bundle/index.js')
    if bundle_bun.is_file() and bundle_js.is_file():
        candidates.append([str(bundle_bun), 'run', str(bundle_js), '--port', str(PORT)])
    npx = shutil.which('npx')
    if npx:
        candidates.append([npx, '--yes', '@codex-ultra/cxu', 'serve', '--port', str(PORT)])
    return candidates


candidates = candidate_commands()
if not candidates:
    raise RuntimeError('cxu was not installed and no fallback runtime was found')

started = False
last_error = None
for service_command in candidates:
    print('$', ' '.join(map(str, service_command)))
    service_log = open(log_path, 'a', encoding='utf-8')
    service_log.write(f"\n$ {' '.join(map(str, service_command))}\n")
    service_log.flush()
    service_process = subprocess.Popen(service_command, cwd=REPO_DIR, env=os.environ.copy(), stdout=service_log, stderr=subprocess.STDOUT, start_new_session=True)
    time.sleep(3)
    status = service_process.poll()
    if status is not None:
        last_error = f"exited early with status {status}"
        print(f'Candidate failed ({last_error}); trying next fallback if any...')
        dump_log_tail()
        service_process = None
        continue
    for _ in range(30):
        if service_process.poll() is not None:
            break
        if port_is_open():
            break
        time.sleep(1)
    if service_process.poll() is not None:
        last_error = f'exited with status {service_process.poll()}'
        print(f'Candidate failed ({last_error}); trying next fallback if any...')
        dump_log_tail()
        service_process = None
        continue
    if port_is_open():
        started = True
        break
    print('Candidate did not open the port; trying next fallback if any...')
    dump_log_tail()
    try:
        service_process.terminate()
    except Exception:
        pass
    service_process = None

if not started:
    dump_log_tail()
    raise RuntimeError(f'Codex Ultra did not start (last error: {last_error}); see {log_path}')

print(f'Codex Ultra is running on http://127.0.0.1:{PORT}')
print(f'Service log: {log_path}')

In [ ]:
def ensure_cloudflared():
    existing = shutil.which('cloudflared')
    if existing:
        return existing
    machine = platform.machine().lower()
    arch = {'x86_64': 'amd64', 'amd64': 'amd64', 'aarch64': 'arm64', 'arm64': 'arm64'}.get(machine)
    if arch is None:
        raise RuntimeError(f'Unsupported CPU architecture for cloudflared: {machine}')
    target = BIN_DIR / 'cloudflared'
    url = f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}'
    run(['curl', '-fL', '--retry', '3', '-o', str(target), url])
    target.chmod(0o755)
    return str(target)


def ensure_ngrok():
    existing = shutil.which('ngrok')
    if existing:
        return existing
    machine = platform.machine().lower()
    asset = {'x86_64': 'amd64', 'amd64': 'amd64', 'aarch64': 'arm64', 'arm64': 'arm64'}.get(machine)
    if asset is None:
        raise RuntimeError(f'Unsupported CPU architecture for ngrok: {machine}')
    archive = Path('/tmp/ngrok.zip')
    target = BIN_DIR / 'ngrok'
    url = f'https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-{asset}.zip'
    run(['curl', '-fL', '--retry', '3', '-o', str(archive), url])
    run(['unzip', '-o', str(archive), 'ngrok', '-d', str(BIN_DIR)])
    target.chmod(0o755)
    return str(target)


def start_tunnel():
    global tunnel_process, public_url
    if TUNNEL_PROVIDER in ('none', 'off', 'disabled'):
        print(f'Tunnel disabled. Local URL: http://127.0.0.1:{PORT}')
        return

    if TUNNEL_PROVIDER == 'cloudflare':
        tunnel_command = [ensure_cloudflared(), 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate']
        url_pattern = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
    elif TUNNEL_PROVIDER == 'localtunnel':
        if shutil.which('npx') is None:
            raise RuntimeError('localtunnel requires Node.js and npx')
        tunnel_command = ['npx', '--yes', 'localtunnel', '--port', str(PORT)]
        url_pattern = re.compile(r'https://[a-z0-9-]+\.loca\.lt')
    elif TUNNEL_PROVIDER == 'ngrok':
        ngrok = ensure_ngrok()
        token = os.environ.get('NGROK_AUTHTOKEN')
        if token:
            subprocess.run([ngrok, 'config', 'add-authtoken', token], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        else:
            print('NGROK_AUTHTOKEN is not set; trying the existing ngrok login/configuration.')
        tunnel_command = [ngrok, 'http', str(PORT), '--log', 'stdout']
        url_pattern = re.compile(r'https://[^\s]+\.ngrok(?:-free)?\.(?:app|com)')
    else:
        raise ValueError('TUNNEL_PROVIDER must be cloudflare, localtunnel, ngrok, or none')

    tunnel_process = subprocess.Popen(tunnel_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    deadline = time.time() + 45
    while time.time() < deadline:
        line = tunnel_process.stdout.readline()
        if line:
            match = url_pattern.search(line)
            if match:
                public_url = match.group(0)
                print(f'Public URL: {public_url}')
                print('Keep this Deepnote session running while you use the URL.')
                return
        elif tunnel_process.poll() is not None:
            break
        time.sleep(0.2)
    raise TimeoutError('The tunnel did not provide a public URL. Check the tunnel process output.')


start_tunnel()

In [ ]:
def cleanup(*_args):
    for process, name in ((tunnel_process, 'tunnel'), (service_process, 'service')):
        if process is not None and process.poll() is None:
            print(f'Stopping {name}...')
            process.terminate()
    if 'service_log' in globals() and not service_log.closed:
        service_log.close()

atexit.register(cleanup)
signal.signal(signal.SIGINT, cleanup)
signal.signal(signal.SIGTERM, cleanup)

print('Launcher is ready.')
print('To stop manually, run: cleanup()')